In [2]:
################################################################################
# Install Project
################################################################################
import sys
import os
import importlib

if sys.version_info >= (3, 12):
    from types import ModuleType
    import types
    # Create a fake 'imp' module in memory
    m = types.ModuleType('imp')
    m.reload = importlib.reload
    sys.modules['imp'] = m

%load_ext autoreload
%autoreload 2

if os.path.isdir("variable_patchtst_project"):
    %cd variable_patchtst_project
    !git pull
    %cd ..
else:

    # Clone your repo
    !git clone https://github.com/Ethan-Russell/variable_patchtst_project.git

from variable_patchtst_project.src.variable_patchtst_project.utils import *
from variable_patchtst_project.src.variable_patchtst_project.models import *

Cloning into 'variable_patchtst_project'...
remote: Enumerating objects: 125, done.
remote: Counting objects: 100% (125/125), done.
remote: Compressing objects: 100% (92/92), done.
remote: Total 125 (delta 39), reused 111 (delta 25), pack-reused 0 (from 0)
Receiving objects: 100% (125/125), 51.76 KiB | 6.47 MiB/s, done.
Resolving deltas: 100% (39/39), done.


In [ ]:
%pip install ptflops

In [3]:
import torch
import json
import pandas as pd
import time

from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
from tqdm.auto import tqdm

## CONNECT TO DEVICE ##
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [4]:
## LOAD DATA ##

# Read the electricity dataset from file
data, df = read_electricity_data()

target_idx = df.columns.get_loc('nat_demand')

# Partition into training and test sets
train_ratio = 0.7
val_ratio = 0.15
total_samples = data.shape[0]
train_end_idx = int(total_samples * train_ratio)
val_end_idx = train_end_idx + int(total_samples * val_ratio)

# Split the data
data_train = data[:train_end_idx, :]
data_val = data[train_end_idx:val_end_idx, :]
data_test = data[val_end_idx:, :]

# Make the scaler according to the training data, then scale both training and test data
scaler = StandardScaler(data_train)
data_train_scaled = scaler.transform(data_train)
data_val_scaled = scaler.transform(data_val)
data_test_scaled = scaler.transform(data_test)



Mounted at /content/drive
DataFrame after datetime conversion, sorting, and adding/removing columns:



,nat_demand,T2M_toc,QV2M_toc,TQL_toc,W2M_toc,T2M_san,QV2M_san,TQL_san,W2M_san,T2M_dav,QV2M_dav,TQL_dav,W2M_dav,hour_sin,hour_cos
0,970.3450,25.865259,0.018576,0.016174,21.850546,23.482446,0.017272,0.001855,10.328949,22.662134,0.016562,0.096100,5.364148,0.258819,0.965926
1,912.1755,25.899255,0.018653,0.016418,22.166944,23.399255,0.017265,0.001327,10.681517,22.578943,0.016509,0.087646,5.572471,0.500000,0.866025
2,900.2688,25.937280,0.018768,0.015480,22.454911,23.343530,0.017211,0.001428,10.874924,22.531030,0.016479,0.078735,5.871184,0.707107,0.707107
3,889.9538,25.957544,0.018890,0.016273,22.110481,23.238794,0.017128,0.002599,10.518620,22.512231,0.016487,0.068390,5.883621,0.866025,0.500000
4,893.6865,25.973840,0.018981,0.017281,21.186089,23.075403,0.017059,0.001729,9.733589,22.481653,0.016456,0.064362,5.611724,0.965926,0.258819



DataFrame info:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48048 entries, 0 to 48047
Data columns (total 15 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   nat_demand  48048 non-null  float64
 1   T2M_toc     48048 non-null  float64
 2   QV2M_toc    48048 non-null  float64
 3   TQL_toc     48048 non-null  float64
 4   W2M_toc     48048 non-null  float64
 5   T2M_san     48048 non-null  float64
 6   QV2M_san    48048 non-null  float64
 7   TQL_san     48048 non-null  float64
 8   W2M_san     48048 non-null  float64
 9   T2M_dav     48048 non-null  float64
 10  QV2M_dav    48048 non-null  float64
 11  TQL_dav     48048 non-null  float64
 12  W2M_dav     48048 non-null  float64
 13  hour_sin    48048 non-null  float64
 14  hour_cos    48048 non-null  float64
dtypes: float64(15)
memory usage: 5.5 MB

Data has been converted into numpy array with shape: (48048, 15)


In [ ]:
## TRAINING LOOP ##
from ptflops import get_model_complexity_info
import random
import numpy as np


def val(model, device, criterion, val_loader):
    # Validation step
    model.eval() # Set model to evaluation mode
    val_loss = 0.0
    with torch.no_grad(): # Disable gradient calculation for validation
        for batch_X_val, batch_y_val in val_loader:
            batch_X_val = batch_X_val.to(device)
            batch_y_val = batch_y_val.to(device)
            pred = model(batch_X_val)
            loss = criterion(pred, batch_y_val)
            val_loss += loss.item()
    avg_val_loss = val_loss / len(val_loader)
    model.train()
    return avg_val_loss

def train(model, device, criterion, optimizer, num_epochs, train_loader, val_loader):
    # Training loop
    print("Starting training...")
    training_loss = []
    validation_loss = []
    model.to(device)
    model.train() # Set model to training mode

    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    train_time = 0.0

    for epoch in range(num_epochs):
        train_loss = 0.0
        train_time_epoch_start = time.perf_counter()
        for batch_X, batch_y in train_loader:
            batch_X = batch_X.to(device)
            batch_y = batch_y.to(device)
            optimizer.zero_grad() # Zero the gradients
            # print(outputs)
            #print(type(outputs))
            #loss = criterion(outputs, batch_y)
            pred = model(
                batch_X,
            )
            loss = criterion(pred, batch_y)

            loss.backward() # Perform backpropagation
            optimizer.step() # Update model parameters
            train_loss += loss.item()
        train_time_epoch_end = time.perf_counter()
        train_time += train_time_epoch_end - train_time_epoch_start

        avg_train_loss = train_loss / len(train_loader)
        training_loss.append(avg_train_loss)

        avg_val_loss = val(model, device, criterion, val_loader)
        validation_loss.append(avg_val_loss)

        print(f'Epoch [{epoch+1}/{num_epochs}], Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}')

    peak_mem_allocated = torch.cuda.max_memory_allocated() / (1024 ** 2)
    peak_mem_reserved = torch.cuda.max_memory_reserved() / (1024 ** 2)
    train_metrics = {
        "train_time": train_time,
        "peak_mem_allocated": peak_mem_allocated,
        "peak_mem_reserved": peak_mem_reserved,
    }
    return training_loss, validation_loss, train_metrics


## GENERATE LOOP ##

def generate(data_loader, model, device, num_latency_runs=200, num_throughput_runs=200):
    model.eval()
    all_preds = []
    all_targets = []
    latencies = []
    total_samples = 0
    is_first = True
    with torch.inference_mode():
        sample, _ = next(iter(data_loader))
        sample = sample.to(device)

        # WARMUP
        for _ in range(200):
            torch.cuda.synchronize()
            output = model(sample)
            torch.cuda.synchronize()

        # LATENCY
        for _ in range(num_latency_runs):
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            start_time = time.perf_counter()
            output = model(sample)
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            end_time = time.perf_counter()

            latencies.append(end_time - start_time)

        avg_latency_ms = (sum(latencies) / len(latencies)) * 1000

        # Throughput: How many samples per second can we churn through?
        torch.cuda.synchronize()
        start = time.perf_counter()
        for _ in range(num_throughput_runs):
            _ = model(sample)

        torch.cuda.synchronize()
        end = time.perf_counter()

        total_time = end - start
        batch_size = sample.shape[0]
        throughput = (num_throughput_runs * batch_size) / total_time # samples/sec

        for data, label in data_loader:
            data = data.to(device)
            label = label.to(device)

            output = model(data)

            all_preds.append(output.cpu())
            all_targets.append(label.cpu())

    all_preds = torch.cat(all_preds, dim=0)
    all_targets = torch.cat(all_targets, dim=0)
    all_preds = all_preds.float()


    return all_targets, all_preds, avg_latency_ms, throughput





## Helper Functions ##

def compute_acc(preds, targets, target_idx, tolerance = 0.1):

    # expects tensor [batch_size, seq_len, features]
    true_vals = targets[:, :, target_idx]
    pred_vals = preds[:, :, target_idx]

    # Uses MSE to compute accuracy
    mse = torch.mean((pred_vals - true_vals) ** 2).item()

    # Uses MAE to compute accuracy
    mae = torch.mean(torch.abs(pred_vals - true_vals)).item()

    # Uses a tolerance to see if the two values are within that. Prediction should be within tolerance % of actual value
    epsilon = 1e-8
    relative_error = torch.abs(pred_vals - true_vals) / (torch.abs(true_vals) + epsilon)
    tolerance_acc = (relative_error <= tolerance).float().mean().item()

    return mse, mae, tolerance_acc

def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def check_numerical_stability(tensor):
    nan_count = torch.isnan(tensor).sum().item()
    inf_count = torch.isinf(tensor).sum().item()
    return nan_count, inf_count

def make_loaders(seq_len, batch_size=32, forecast_length=24):
    tr = DataLoader(
        ElectricityLoadDataset(data_train_scaled, seq_len, forecast_length, target_idx),
        batch_size=batch_size, shuffle=True
    )
    va = DataLoader(
        ElectricityLoadDataset(data_val_scaled, seq_len, forecast_length, target_idx),
        batch_size=batch_size, shuffle=False
    )
    te = DataLoader(
        ElectricityLoadDataset(data_test_scaled, seq_len, forecast_length, target_idx),
        batch_size=batch_size, shuffle=False
    )
    return tr, va, te

def seed_everything(seed=49):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def run(patch_algo, seq_len, forecast_len, d_model, device, epochs=10, batch_size=32, tolerance=0.1, seed=49):
    seed_everything(seed=seed)
    tr_loader, va_loader, te_loader = make_loaders(seq_len, batch_size=batch_size, forecast_length=forecast_len)
    model = SimplePatchTST(
        patch_algorithm=patch_algo,
        sequence_length=seq_len,
        forecast_length=forecast_len,
        d_model=d_model,
    )
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    tr_losses, val_losses, tr_metrics = train(
        model, device, criterion, optimizer, epochs, tr_loader, va_loader
    )
    targets, preds, latency, throughput = generate(te_loader, model, device)

    # Check numerical stability
    nan_count, inf_count = check_numerical_stability(preds)

    mse, mae, tol_acc = compute_acc(preds, targets, target_idx, tolerance=tolerance)

    # Inference memory (approximate peak during inference)
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    with torch.no_grad():
        for data, _ in te_loader:
            data = data.to(device)
            _ = model(data)
            break  # Just one batch for memory estimate
    inf_peak_mem = torch.cuda.max_memory_allocated() / (1024 ** 2)

    total_samples = len(te_loader.dataset)

    n_macs, n_params = get_model_complexity_info(model, (seq_len, data_train_scaled.shape[1]), print_per_layer_stat=False, verbose=False, as_strings=False)
    model_size_mb = n_params * 4 / (1024 ** 2)  # approx float32

    return {
        "n_params": n_params,
        "n_macs": n_macs,
        "model_size_mb": model_size_mb,
        "mse": mse,
        "mae": mae,
        "tol_acc": tol_acc,
        "best_val_loss": min(val_losses) if len(val_losses) > 0 else -1,
        "train_time_s": tr_metrics["train_time"],
        "peak_train_mem_mb": tr_metrics["peak_mem_allocated"],
        "peak_train_mem_reserved_mb": tr_metrics["peak_mem_reserved"],
        "latency_per_batch_ms": latency,
        "throughput_samples_per_s": throughput,
        "inf_peak_mem_mb": inf_peak_mem,
        "nan_count": nan_count,
        "inf_count": inf_count,
    }

In [ ]:
repo_dir = 'variable_patchtst_project'
drive_dir = 'drive/MyDrive/variable-patchtst-experiments'
def run_configs_and_output(run_id):

    timestamp = time.strftime("%Y%m%d-%H%M%S")

    df = pd.read_csv(f"{repo_dir}/config/configs-{run_id}.csv")
    numeric_cols = df.select_dtypes(include=['number']).columns

    df[numeric_cols] = df[numeric_cols].fillna(-1).astype(int)

    for row in tqdm(df.itertuples(), total=len(df)):
        print(f"Running row: {row.Index}")
        # if 7 <= row.Index <= 27:
            # continue
        algo_name = row.patch_algo_name
        if pd.isna(algo_name):
            print("Skipped empty row")
            continue
        elif algo_name == "uniform":
            algo = UniformPatchAlgorithm(
                row.patch_size,
                row.patch_stride
            )
        elif algo_name == "variable":
            algo = VariablePatchAlgorithm(
                row.min_patch_size,
                row.num_patch_sizes,
                row.num_patches
            )
        else:
            RuntimeError(f"Algorithm name {algo_name} not defined!")

        try:
            res = run(
                patch_algo = algo,
                seq_len = row.seq_len,
                forecast_len = row.forecast_len,
                d_model = row.d_model,
                epochs = row.epochs,
                device = device,
                batch_size = row.batch_size
            )
        except KeyError as e:
            print(f"Skipping row due to Keyrror: {e}")
            continue

        df.loc[row.Index, res.keys()] = res.values()

        # Create output folder if necessary
        os.makedirs(f'{drive_dir}/output', exist_ok=True)

        df.to_csv(f'{drive_dir}/output/outputs-{run_id}-{timestamp}.csv')

In [8]:
config = "32-512"

run_configs_and_output(config)


  0%|          | 0/17 [00:00<?, ?it/s]

Running row: 0
Num patches: 32
Num pad: 16
Starting training...
Epoch [1/20], Train Loss: 0.3825, Val Loss: 0.3956
Epoch [2/20], Train Loss: 0.3596, Val Loss: 0.3530
Epoch [3/20], Train Loss: 0.3562, Val Loss: 0.3459
Epoch [4/20], Train Loss: 0.3539, Val Loss: 0.3686
Epoch [5/20], Train Loss: 0.3522, Val Loss: 0.3434
Epoch [6/20], Train Loss: 0.3526, Val Loss: 0.3520
Epoch [7/20], Train Loss: 0.3526, Val Loss: 0.3561
Epoch [8/20], Train Loss: 0.3519, Val Loss: 0.3292
Epoch [9/20], Train Loss: 0.3530, Val Loss: 0.3262
Epoch [10/20], Train Loss: 0.3517, Val Loss: 0.3653
Epoch [11/20], Train Loss: 0.3528, Val Loss: 0.3879
Epoch [12/20], Train Loss: 0.3528, Val Loss: 0.3490
Epoch [13/20], Train Loss: 0.3529, Val Loss: 0.3645
Epoch [14/20], Train Loss: 0.3512, Val Loss: 0.3539
Epoch [15/20], Train Loss: 0.3520, Val Loss: 0.3289
Epoch [16/20], Train Loss: 0.3518, Val Loss: 0.3371
Epoch [17/20], Train Loss: 0.3520, Val Loss: 0.3348
Epoch [18/20], Train Loss: 0.3520, Val Loss: 0.3393
Epoch [19